# 1-2 MRIの原理と画像の周波数

範囲1 医用画像の種類と原理｜第3回

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/Medical_Imaging_Seminar_Public/blob/main/hiroshima/1_modalities/1-2_mri.ipynb)

## この回で分かるようになること

- MRI装置の構成（静磁場磁石、傾斜磁場コイル、RFコイル、受信コイル）と、水素原子核の共鳴から信号を得るしくみを説明できる
- 周波数エンコードと位相エンコードで信号に位置の情報を入れ、k空間に並べるしくみを説明できる
- MRIで収集するデータ（k空間）と画像が、フーリエ変換で結び付いていることを説明できる
- T1強調画像・T2強調画像の違いを説明できる
- スピンエコー法、グラディエントエコー法、SE-EPIのエコーの作り方と使い分けを説明できる
- 拡散強調像とTOF血管画像が、それぞれどの物理現象を画像にしているかを説明できる
- ガドリニウム造影剤がT1を短くするしくみと、肝細胞特異性造影剤などの特殊な造影剤の特徴を説明できる
- 高速撮像法（高速スピンエコー法、ハーフフーリエ法、HASTE、パラレルイメージング）のしくみと、静磁場の強さによる違いを説明できる
- k空間の一部を削ると画像がどう変わるかを、シミュレーションを使って説明できる

## 使うデータ

- TCIA Pseudo-PHI-DICOM-Data（CC BY 4.0）の腹部MRI：T2強調画像（HASTE）とT1強調画像（FLASH）。同じ患者の画像です
- 初めて実行するときに、`data/` へ自動でダウンロードします（数十秒〜数分）

## 0. 準備

**Colab で開いた場合**: 上の「Open In Colab」ボタンから開き、次のセル（Colab用の準備）を実行します。`pydicom` と `idc-index` の追加インストールと、データ取得に使う `course_data.py` の取得を行います。1〜2分かかります。

**VS Code で開いた場合**: ノートブック右上の「カーネルの選択」で `.venv` を選びます。Colab用の準備セルは、そのまま実行しても何も起きません（`uv sync` で環境が整っているため）。

In [ ]:
import sys
import subprocess
import urllib.request

# Colab では、足りないライブラリと course_data.py（データ取得用）を用意する
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydicom", "idc-index"], check=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/course_data.py", "course_data.py"
    )
    print("準備できました")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pydicom
from course_data import fetch

## 1. 解説

### 1-1. MRI装置のしくみと信号の源

MRI（magnetic resonance imaging、磁気共鳴画像）は、体の中の水素原子核が特定の周波数の電波に共鳴する現象を使って画像を作ります。X線やCTは体を透過したX線を測りますが、MRIは体の中の水素原子核が出す電波を受信します。

#### 装置の構成

![Schematic of an MRI scanner](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_scanner_schematic.png)

MRI装置は、主に次の部品でできています。図の左は装置を横から見たところ、右は穴（ボア）の正面から見た断面です。

- **静磁場磁石**：ドーナツ形の本体の大部分を占める磁石で、穴の中に強く均一な磁場を作ります。磁場の向きは穴の奥行き方向で、この方向をz方向と呼びます。臨床用の装置は1.5テスラか3テスラが主流で（磁場の強さによる違いは1-8で扱います）、地球の磁場（約0.00005テスラ）の3万〜6万倍にあたります。多くの装置は、液体ヘリウムで−269℃近くまで冷やした超伝導コイルに電流を流し続けて磁場を作るので、検査をしていない時間も磁場は常にかかっています
- **傾斜磁場コイル**：磁石の内側にあるコイルで、x、y、zの3方向それぞれについて、磁場の強さを位置によって少しずつ変えます（傾斜磁場）。どの位置から来た信号かを区別するために使います（1-2で説明します）。撮影中の大きな音は、このコイルの電流を高速で切り替えるときに、コイルが磁場から力を受けて振動して出る音です
- **RF送信コイル**：水素原子核を共鳴させる電波（RFパルス、radio frequency pulse）を体に当てます。装置に組み込まれた大きなコイル（ボディコイル）を使うことが多くなっています
- **受信コイル**：水素原子核が出す弱い電波を受け取るアンテナです。頭部用のコイルや、体の表面に載せる表面コイルのように、撮影する部位にできるだけ近づけて置きます。体に近いほど、雑音に対する信号の比（SN比）が高くなるからです。現在は、小さなコイルを多数並べたアレイコイルが主流です
- **シールドルーム**：MRIで使う電波は、FMラジオに近い数十〜百数十MHzの周波数です。外からの電波が混ざると画像に雑音が出るので、検査室全体を金属で囲んで外の電波を遮ります

![Siemens MAGNETOM Flow.Elite 1.5 T MRI system](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/photo_mri_1_5t.jpg)

実際のMRI装置の例（Siemens Healthineers MAGNETOM Flow.Elite、1.5 T）です。白いカバーの中に、模式図の超伝導磁石、傾斜磁場コイル、RF送信コイルが同心円状に収められています。CTより奥行きが長く、穴（ボア）の中に体の大部分が入ります。検査室の壁と天井の風景は、閉所の圧迫感を和らげるための工夫です。検査室全体がシールドルームになっています。

出典: Wikimedia Commons — [Clinical 1.5T MRI system.jpg](https://commons.wikimedia.org/wiki/File:Clinical_1.5T_MRI_system.jpg) ／ 撮影: FbrG ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0)

![Siemens MRI with a head coil and surface coils](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/photo_mri_coils.jpg)

寝台の上に受信コイルを置いたMRI装置です。中央の籠のような形のものが頭部用コイル、手前の平たいものが体の表面に載せる表面コイルです。撮影する部位に合わせてコイルを選び、体にできるだけ近づけて置きます。

出典: Wikimedia Commons — [IRM siemens avec antennes.jpg](https://commons.wikimedia.org/wiki/File:IRM_siemens_avec_antennes.jpg) ／ 撮影: Raziel（フランス語版Wikipedia） ／ ライセンス: [CC BY 2.5](https://creativecommons.org/licenses/by/2.5)

出典（模式図）: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 水素原子核とラーモア周波数

体の大部分は水と脂肪でできていて、その中に水素原子核（陽子1個）が大量に含まれます。水素原子核はスピンという性質をもち、小さな磁石のように振る舞います。

磁場のない所では、水素原子核の向きはばらばらで、全体としての磁石の性質は打ち消し合っています。強い磁場の中に置くと、磁場の向きにそろう原子核がわずかに多くなり、全体として磁場の方向を向いた磁化（正味の磁化 $M$）が生じます。多い分は1.5テスラでも100万個に5個程度しかありませんが、体には水素原子核が膨大な数あるので、測れる大きさの磁化になります。

磁場の中の水素原子核は、コマが首を振るように、磁場の向きを軸にして回転しています（歳差運動）。この回転の周波数をラーモア周波数と呼び、磁場の強さ $B_0$ に比例します。

$$f_0 = \bar\gamma\,B_0,\qquad \bar\gamma \approx 42.58\ \mathrm{MHz/T}$$

$\bar\gamma$ は水素原子核に固有の定数（磁気回転比を $2\pi$ で割った値）です。ラーモア周波数は、1.5テスラで約64MHz、3テスラで約128MHzになります。2-1で使うDICOMファイルにも、撮影したときの磁場の強さと共鳴周波数が記録されているので、この式が成り立つことを確かめます。

#### RFパルスで磁化を倒す

ラーモア周波数と同じ周波数の電波（RFパルス）を当てると、水素原子核は電波からエネルギーを受け取ります。これが共鳴で、周波数の違う電波ではほとんど影響を受けません。正味の磁化で見ると、RFパルスを当てている間、磁化はラーモア周波数で回転しながら、z軸から少しずつ倒れていきます。

![Excitation by an RF pulse, precession and free induction decay](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_excitation_fid.png)

(a)は、磁化をちょうど90°倒すRFパルス（90°パルス）を当てている間の、磁化の先端の動きです。倒れる角度（フリップ角）は、RFパルスの強さと長さで決まります。90°パルスは縦磁化（z方向の成分）をすべて横磁化（xy平面の成分）に変え、180°パルスは磁化を反対向きにします。図では見やすいように6回転で倒していますが、実際には数ミリ秒のパルスの間に数十万回転します。

#### 受信コイルで信号を受け取る

RFパルスを止めると、倒れた磁化（横磁化）はxy平面の中でラーモア周波数で回転し続けます（(b)）。回転する磁石の近くにコイルを置くと、電磁誘導によってコイルに電圧が生じます。発電機と同じしくみで、受信コイルはこの電圧をMRIの信号として受け取ります（(c)）。

横磁化は時間とともに小さくなるので（1-3で説明するT2緩和とT2*）、信号も振動しながら減衰します。この信号を自由誘導減衰（FID、free induction decay）と呼びます。(c)の振動は見やすく描いたもので、実際には1.5テスラで1秒間に約6400万回振動します。装置は受信した信号から、ラーモア周波数を基準にしたわずかな周波数のずれ（数kHz〜数十kHz）だけを取り出して記録します。1-2で説明する位置の情報は、このずれに入っています。

出典: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 正常な頭部のMRI画像の例

![MRI of a normal head](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_head_brain_normal.jpg)

上の画像は正常な頭部を水平に切断した像（axial）です。明るい画素は、その場所の水素原子核から戻ってきた信号が強いことを意味します。この画像では皮下や眼窩の脂肪、眼の中のガラス体（主成分は水）が明るく写っています。

出典: Wikimedia Commons — [MRI Head Brain Normal](https://commons.wikimedia.org/wiki/File:MRI_Head_Brain_Normal.jpg) ／ 作者: Ptrump16 ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0)

#### MRIの特徴と安全上の注意

X線を使わないので被ばくはありません。そのかわり、撮影に時間がかかります（理由は1-2で説明します）。安全の面では、次の点に注意が必要です。

- 静磁場は常にかかっているので、鉄を含む物（はさみ、酸素ボンベ、車いすなど）を検査室に持ち込むと、磁石に強く引き寄せられて飛び、事故になります
- 心臓ペースメーカーなどの体内の医療機器や金属は、種類や条件によって撮影できない場合があります
- RFパルスのエネルギーの一部は体に吸収されて熱になります。装置は、体重あたりに吸収されるエネルギー（SAR、比吸収率）が上限を超えないように、当てるRFパルスの量を制限しています

### 1-2. 位置の情報を信号に入れる方法とk空間

受信コイルは、断面のあらゆる場所から出る電波をまとめて受け取ります。受け取った電波そのものには、どこから出たかの情報は含まれていません。MRIでは、傾斜磁場を使って位置によって共鳴周波数や位相を変え、信号に位置の情報を入れます。この操作をエンコード（符号化）と呼び、次の3つを組み合わせます。

| 操作 | 傾斜磁場をかけるとき | 位置の情報の入れ方 | 決まる方向 |
|---|---|---|---|
| スライス選択 | RFパルスを当てている間 | 1枚の断面だけを共鳴させる | 断面の厚さの方向 |
| 位相エンコード | RFパルスのあと、信号を読む前 | 位置によって位相をずらす | 画像の一方の方向（位相方向） |
| 周波数エンコード | 信号を読み取っている間 | 位置によって周波数を変える | 画像のもう一方の方向（周波数方向） |

ここでは、断面の厚さの方向をz、周波数方向をx、位相方向をyとして説明します。実際には、どの方向をどの役割に使うかを撮影のたびに選べます。

#### スライス選択

z方向に傾斜磁場をかけると、磁場の強さがzの位置によって少しずつ変わり、ラーモア周波数も位置によって変わります。このとき、ある幅の周波数だけを含むRFパルスを当てると、その周波数で共鳴する位置の水素原子核だけが倒れます。

![Slice selection with a z gradient](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_slice_selection.png)

図は、1.5テスラの装置で10 mT/mの傾斜磁場をかけた例です。1 mm進むごとに共鳴周波数が約0.43 kHz変わるので、基準から10.6〜14.9 kHzずれた周波数だけを含むRFパルスを当てると、z = 25〜35 mmの厚さ10 mmの断面だけが共鳴します。RFパルスの中心の周波数を変えれば断面の位置が、周波数の幅や傾斜磁場の強さを変えれば断面の厚さが変わります。

#### 周波数エンコード

選んだ断面の中で、信号を読み取っている間にx方向の傾斜磁場をかけます。すると、xの位置によってラーモア周波数が変わり、それぞれの位置の水素原子核が違う周波数で信号を出します。

![Frequency encoding and Fourier transform](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_frequency_encoding.png)

(1)は、x方向に並んだ3本の水の管です。上の目盛りは、5 mT/mの傾斜磁場をかけたときに、それぞれの位置から出る信号の周波数を示しています。受信コイルが受け取るのは、すべての位置の信号を足し合わせた(2)の波形です。この波形をフーリエ変換して周波数ごとの成分に分け、周波数を位置に読み替えると、(3)のようにx方向の水の分布が得られます。管の上辺の細かい波打ちは、信号を有限の点数しか記録しないために生じるもの（打ち切りアーチファクト）です。

#### 位相エンコード

周波数エンコードで分けられるのは、x方向だけです。x方向とy方向の傾斜磁場を同時にかけても、磁場の傾きが斜めの1方向になるだけで、xとyを区別できません。そこでy方向には、位相を使います。

RFパルスのあと、信号を読み取る前に、y方向の傾斜磁場を短時間だけかけます。かけている間はyの位置によって回転の速さが変わるので、傾斜磁場を止めたときには、yの位置に比例して位相がずれた状態になります。

![Phase encoding steps](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_phase_encoding.png)

図は、y方向に並んだ8か所の横磁化の向き（位相）を時計の針のように描いたもので、背景の明るさは位相のcosです。ステップ0では傾斜磁場をかけないので、すべての位置で位相がそろっています。ステップ1では撮影範囲（FOV、field of view）全体で位相が1回転、ステップ2では2回転ずれます。位相のずれ方は、y方向の縞模様になっています。

1回の読み取りで受け取る信号は全位置の信号の和なので、1回だけでは位置ごとの位相を区別できません。そこで、y方向の傾斜磁場の強さを1ステップずつ変えながら、RFパルスと読み取りを何度も繰り返します。ステップkで得られる信号は、「FOV全体でk回転する縞模様」がこの断面にどれだけ含まれるかを表します。こうして集めた信号の組をフーリエ変換すると、y方向の分布が得られます。

位相方向に256画素の画像を作るには、位相エンコードを256ステップ繰り返します。1ステップごとに、RFパルスを繰り返す間隔（TR、repetition time。1-3で詳しく扱います）の時間がかかるので、たとえばTRが500 msなら、500 ms × 256 ≒ 2分かかります。MRIの撮影に時間がかかるのは、主にこのためです。実際の装置では、1回のRFパルスのあとに複数のステップを続けて読み取る高速撮像法（高速スピンエコー法やEPIなど）で時間を短くしています。

#### 集めた信号がk空間に並ぶ

1回の読み取り（周波数エンコード）で記録した信号の列が、k空間の1行になります。位相エンコードのステップを変えるたびに、別の行が埋まります。

![Filling k-space row by row](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_kspace_filling.png)

k空間の横方向（kx）は周波数方向、縦方向（ky）は位相方向に対応します。すべての行がそろったら、2次元のフーリエ変換で画像に戻します。

出典（この項の図）: 自作（NumPy と matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### k空間とフーリエ変換

ここまでの操作をまとめると、MRI装置は体の各点の信号を1点ずつ直接測っているのではありません。傾斜磁場を使って、画像に含まれる「縞模様の成分」を1つずつ測っています。この「縞模様の成分」とフーリエ変換の関係を、1次元から順に見ていきます。

##### 1次元：分布を波の足し合わせで表す

どんな分布も、撮影範囲（FOV）の中で0回、1回、2回…と振動する波（cosの波）を、それぞれ適当な大きさ（振幅）とずらし方（位相）で足し合わせると表せます。k回振動する波を「空間周波数 k の成分」と呼び、分布からそれぞれの成分の振幅と位相を求める計算がフーリエ変換、成分を足し合わせて分布に戻す計算が逆フーリエ変換です。

![Building a 1D profile from waves](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_fourier_1d.png)

1-2の周波数エンコードの図と同じ、3本の水の管の分布を使った例です。

- (1) 空間周波数1〜3の成分です。どの成分も単純な波ですが、振幅と位相（山の位置）がそれぞれ違います。破線は空間周波数0の成分で、分布全体の平均の高さを表します
- (2) 成分ごとの振幅を並べたもので、これが1次元のk空間にあたります。低い空間周波数ほど振幅が大きくなっています
- (3)〜(6) 空間周波数0から順に成分を足し合わせたものです。低い成分だけではおおまかな形しか表せず、細かい成分を足すほど、管のふちのような急な変化が再現されます。成分を途中で打ち切ると、ふちの近くに細かい波打ちが残ります（1-2の周波数エンコードの図で見た打ち切りアーチファクトと同じ現象です）

式で書くと、画素 $x$（$x = 0, 1, \ldots, N-1$）の値 $\rho(x)$ と、空間周波数 $k$ の成分 $S(k)$ は、次の関係にあります。

$$S(k) = \sum_{x=0}^{N-1} \rho(x)\, e^{-i 2\pi k x / N}, \qquad \rho(x) = \frac{1}{N}\sum_{k} S(k)\, e^{\,i 2\pi k x / N}$$

$e^{i\theta} = \cos\theta + i\sin\theta$ なので、$S(k)$ は複素数で、その絶対値が成分の振幅、角度が成分の位相を表します。

##### MRIの信号はそのままフーリエ変換になっている

周波数エンコードでは、傾斜磁場をかけてから時間 $t$ がたつと、位置 $x$ の水素原子核の位相が $2\pi \bar\gamma G_x x t$ だけ進みます。受信コイルはすべての位置の信号を足し合わせて受け取るので、時刻 $t$ の信号は

$$s(t) = \sum_x \rho(x)\, e^{-i 2\pi k_x x}, \qquad k_x = \bar\gamma\, G_x\, t$$

となり、上のフーリエ変換の式と同じ形になります（符号は位相の約束によります）。つまり、読み取りの時間が進むにつれて $k_x$ が増え、受け取った信号の列がそのままk空間の1行になります。位相エンコードでは、傾斜磁場の強さと時間の積で $k_y$ を決めます。2次元では

$$S(k_x, k_y) = \sum_{x}\sum_{y} \rho(x, y)\, e^{-i 2\pi (k_x x + k_y y)}$$

で、MRIで集めるデータは、画像の2次元フーリエ変換そのものです。

##### 2次元：k空間の1点は1つの縞模様

![Each point in k-space corresponds to a stripe pattern](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_fourier_2d.png)

2次元では、k空間の1点 $(k_x, k_y)$ が1つの縞模様に対応します。原点からの向きが縞の向き（縞は $(k_x, k_y)$ の方向に並ぶ）を、原点からの距離が縞の細かさを決めます。画像は実数なので、原点について対称な2点（$(k_x, k_y)$ と $(-k_x, -k_y)$）は同じ縞を表し、値は互いに複素共役になります。この性質は、1-7のハーフフーリエ法で使います。

k空間と画像のあいだには、次の関係があります。

- **中心**：k空間の中心（$k = 0$）の値は、すべての画素の値の合計です。中心の近くの粗い縞の成分が、画像全体の明るさとコントラストを決めます
- **周辺**：周辺の細かい縞の成分が、輪郭や細かい構造を表します
- **k空間の範囲と画素の大きさ**：集めたk空間の範囲が広いほど細かい縞まで表せるので、画素が小さく（分解能が高く）なります
- **k空間の間隔と撮影範囲**：k空間の点の間隔 $\Delta k$ と撮影範囲（FOV）のあいだには、$\Delta k = 1/\mathrm{FOV}$ の関係があります

では、k空間の中心と周辺は、それぞれ画像のどんな特徴を担っているのでしょうか。この回の「動かしてみる」で、k空間の一部だけを残して画像に戻し、確かめます。

なお、ここでは実際の生データの代わりに、完成した画像をフーリエ変換して作ったk空間を使います。原理を確かめるためのシミュレーションです。

出典（この項の図）: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 画像の周波数方向と位相方向

MRIの画像には、周波数エンコードをした方向（周波数方向）と、位相エンコードをした方向（位相方向）があります。画像を見ただけでは区別できませんが、次の点で性質が違います。

- **撮影時間**：位相方向の画素数（ステップ数）だけ繰り返しが必要なので、撮影時間は位相方向の画素数に比例します。周波数方向の画素数を増やしても、1回の読み取りで記録する点数が増えるだけなので、撮影時間はほとんど変わりません。そのため、撮影範囲が長方形のときは短い辺を位相方向にして、ステップ数を減らすことがよくあります
- **体の動きによるアーチファクト**：1回の読み取りは数msで終わりますが、位相エンコードのステップはTRごとに、全体で数秒〜数分かけて集めます。撮影中に呼吸や心拍、血管の拍動で体が動くと、位置のずれたデータがステップごとに混ざり、その影響は位相方向に広がります。動きが周期的だと、元の像のコピー（ゴースト）が位相方向に並びます（理由は次の項で説明します）
- **折り返し**：撮影範囲の外にある体の部分が、位相方向の反対側に折り返して写ることがあります（折り返しアーチファクト）。周波数方向では、装置が撮影範囲の外に当たる周波数を取り除くので起こりにくくなります
- **化学シフト**：脂肪の水素原子核は、水の水素原子核より共鳴周波数がわずかに低く（約3.5 ppm、1.5テスラで約220 Hz）、周波数エンコードでは実際と少しずれた位置に写ります。このずれは周波数方向に生じます
- **EPIの歪み**：1回の励起で画像全体のデータを集める高速撮像法（EPI。1-5の拡散強調像で使います）では、磁場のずれによる位置のずれが、位相方向に大きく生じます（理由は1-4のSE-EPIの項で説明します）

![Motion ghosts appear along the phase-encoding direction](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_motion_ghost.png)

図は、周期的に上下に動く模型（Shepp-Loganファントム）を、位相エンコードの方向を変えて撮影したシミュレーションです。中央と右で動きはどちらも上下ですが、ゴーストは位相エンコードの方向（PE）に並びます。アーチファクトが並ぶ向きは、動きの向きではなく位相方向で決まることが分かります。

2-1で使う腹部の画像では、どちらが位相方向かがDICOMのタグ（InPlanePhaseEncodingDirection）に記録されています。

出典: 自作（scikit-image の Shepp-Logan ファントムを使い、NumPy でシミュレーション。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 動きによるアーチファクトが位相方向に出る理由

上の図で、ゴーストが動きの向きではなく位相方向に並んだのは、周波数方向と位相方向で、データを集める時間の長さが大きく違うからです。

- **周波数方向**：k空間の1行（周波数エンコード）は、1回の読み取りで数ミリ秒のうちに集めます。この間に体はほとんど動かないので、1行の中のデータは、どれも同じ位置の体から得られたものとしてそろっています
- **位相方向**：行と行（位相エンコードのステップ）は、TRごとに1行ずつ、全体で数秒〜数分かけて集めます。その間に呼吸や心拍で体が動くと、行ごとに少しずつ違う位置の体を測ったことになります

![How motion causes ghosts along the phase-encoding direction](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_motion_kspace.png)

図は、ステップごとに上下にずれる模型（前の図の中央と同じ条件）で、k空間と画像に何が起きるかを示したものです。

- (a) ステップごとの体の位置です。1回の読み取りの間はほぼ止まっていますが、ステップが進むと位置が周期的に変わります
- (b) 動きのためにk空間の各点に生じた位相のずれです。フーリエ変換には「画像が $d$ だけずれると、k空間には $e^{-i2\pi k_y d}$ という位相がかかる」という性質（シフト定理）があります。ずれ $d$ は行ごとに違うので、位相のずれは横（$k_x$ の方向）には一定で、縦（$k_y$ の方向）にだけ変わる縞になります
- (c) 画像の上下方向（位相方向）の信号の分布です。k空間で $k_y$ の方向にだけ乱れたデータを逆フーリエ変換すると、その乱れは画像の $y$ の方向、つまり位相方向にだけ広がります。動きが周期的なので、乱れも $k_y$ の方向に周期的になり、元の像のコピー（ゴースト）が一定の間隔で並びます

位相方向を左右にした前の図の右側でも同じことが起こります。動きは上下でも、データを時間をかけて集めるのは左右の方向なので、ゴーストは左右に並びます。

周期的な動きによるゴーストの間隔は、次の式で見積もれます。

$$\text{ゴーストの間隔（画素）} = \frac{\text{位相エンコード数} \times \mathrm{TR} \times \text{加算回数}}{\text{動きの周期}}$$

図の条件では、256ステップで8ステップごとに1周期なので、間隔は $256/8 = 32$ 画素です。たとえば、TR 500 msで256ステップを集めるときに呼吸の周期が4秒なら、ゴーストの間隔は $256 \times 0.5 / 4 = 32$ 画素になります。心拍（周期約1秒）や大動脈の拍動では、間隔は128画素と大きくなります。動きが周期的でないと、ゴーストは位相方向のぼけ（にじみ）として現れます。

動きによるアーチファクトを減らすには、次のような方法があります。

- 息止めでの撮影、呼吸や心電図に合わせて決まった時期のデータだけを集める同期撮影
- 動く部分（腹壁など）の信号をあらかじめ消しておく飽和パルス
- 位相方向を変えて、ゴーストが見たい部分に重ならないようにする
- 1回の励起で画像全体を集める高速撮像法（1-7）で、動きが問題にならないほど短時間で撮る
- k空間を放射状や回転する帯状に集め、中心を何度も測って動きを補正する方法（PROPELLER、BLADEなど）

出典: 自作（NumPy によるシミュレーションと matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-3. T1強調画像とT2強調画像

RFパルスで傾けた水素原子核の向きは、電波を止めると元の状態へ戻っていきます。この戻る過程を**緩和**と呼び、どの方向の成分に注目するかで2種類に分けます。

- **T1緩和（縦緩和）**：磁場の方向の成分（縦磁化）が回復していく過程。脂肪では速く、水ではゆっくり進む
- **T2緩和（横緩和）**：磁場に垂直な方向の成分（横磁化）が減っていく過程。脂肪では速く、水ではゆっくり進む

T1とT2は、それぞれの緩和の速さを表す時間で、組織ごとに値が違います。MRIの画像の明るさの差は、主にこの違いから生まれます。T1とT2のどちらの違いを画像に強く反映させるかは、撮影の条件で選びます。条件の中心になるのは、次の2つの時間です。

- **TR（repetition time、繰り返し時間）**：RFパルスを当ててから、次のRFパルスを当てるまでの間隔。1-2で見たように、1枚の画像を作るには位相エンコードのステップを変えながらk空間の行を何十〜何百本も集めるので、RFパルスを何度も繰り返し当てます
- **TE（echo time、エコー時間）**：RFパルスを当ててから、信号を読み取るまでの時間

| | TR | TE | 水（脳脊髄液、尿、胆汁など） | 脂肪 |
|---|---|---|---|---|
| T1強調画像 | 短い（数百ms） | 短い（十数ms） | 暗い | 明るい |
| T2強調画像 | 長い（数千ms） | 長い（100ms前後） | 明るい | 明るい〜中間（撮像法による） |

TRとTEの値は、スピンエコー法で撮影する場合の目安です。なぜこの組み合わせでそれぞれの強調画像になるのかは、このあとのグラフで確かめます。

X線やCTと違い、MRIの画素値には「水が0」のような決まった基準がありません。同じ組織でも、装置や撮影条件によって値が変わります。このことは、範囲3で画像から特徴量を取り出すときに問題になります。

#### T1とT2が組織ごとに違う理由

T1緩和とT2緩和は、RFパルスを止めたあとに同時に進みますが、進むしくみが違います。

**T1緩和（スピン格子緩和）**では、RFパルスで受け取った余分なエネルギーを、スピンが周囲の分子の集まり（格子）へ渡します。受け渡しのきっかけは、分子が動くことで生じる局所磁場の揺らぎです。隣り合う原子核はそれぞれ小さな磁石なので、分子が回転したり移動したりすると、その場所の磁場が細かく揺れます。この揺れの速さが、ラーモア周波数（1-1。1.5テスラで約64MHz）に近いほど、エネルギーの受け渡しが起こりやすくなります。

- 自由に動ける水の分子は、この周波数よりはるかに速く動くので、受け渡しが起こりにくく、T1が長くなります
- 脂肪の分子は大きく、動きがゆっくりで、揺れの速さがこの周波数に近いので、T1が短くなります

T1時間は、縦磁化が最大の63%まで回復するまでの時間と定義されます。

**T2緩和（スピン-スピン緩和）**は、エネルギーを格子へ渡さなくても進みます。RFパルスの直後、横に倒れたスピンは位相（回転の角度）をそろえて回っています。ところが、周囲の原子核が作る局所磁場は場所ごとにわずかに違うので、スピンごとに回転の速さがずれ、時間がたつと位相がばらばらになります（dephasing、位相がずれること）。向きのそろわなくなった横磁化は互いに打ち消し合い、合計が小さくなります。

- 水の分子は速く動くので、局所磁場の違いが平均されて打ち消し合い、位相がずれにくく、T2が長くなります
- 脂肪や、たんぱく質に結び付いた水のように動きの遅い分子では、局所磁場の違いが平均されずに残るので、T2が短くなります

T2時間は、横磁化が最大の37%まで減衰するまでの時間と定義されます。T1緩和で縦磁化が元に戻れば横磁化も残らないので、T2がT1より長くなることはありません。

局所磁場の違いのほかに、装置の磁場の不均一や、組織と空気の境界で生じる磁場の乱れでも位相はずれます。これらを合わせた見かけの減衰時間をT2*（ティーツースター）と呼び、T2より短くなります。場所ごとに一定の磁場のずれによる位相のずれは、スピンエコー法（180°のRFパルスで位相の進みを反転させ、ずれを巻き戻す撮像法）で打ち消せます。グラディエントエコー法（180°パルスを使わず、傾斜磁場で信号を作る撮像法）では打ち消せないので、信号はT2*で減衰します。この違いは、1-4で扱うスピンエコー法とグラディエントエコー法の違いにつながります。

#### 縦磁化の回復：T1緩和のグラフ

縦磁化を横へ完全に倒すRFパルス（90°パルス）を止めた直後を時刻0として、縦磁化 $M_z$（磁場方向の成分）が回復する様子を示します。横軸はT1を単位にした時間、縦軸は最大の縦磁化 $M_0$ で割った値です。回復は $M_z(t)/M_0 = 1 - e^{-t/T_1}$ という指数関数に従い、時刻T1で63%、T1の2倍で86%、3倍で95%まで戻ります。

![T1 recovery curve of longitudinal magnetization](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/t1_recovery_mz.png)

出典: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`。`uv run python instructor/make_figures_1_2_mri.py` で作り直せます）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### T1強調画像：TRを短くして、縦磁化の回復の差を写す

上のグラフは横軸をT1の単位で測っているので、どの組織でも同じ曲線になります。実際の時間（ミリ秒）で描くと、T1の短い組織と長い組織で立ち上がりが大きく違います。次の図は、脂肪（T1 約260ms）と脳脊髄液（T1 約4000ms）の縦磁化の回復を並べたものです。T1の値は1.5テスラでのおおよその値で、文献や測定条件によって変わります。

![Longitudinal recovery of fat and CSF with short and long TR](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/t1_contrast_tr.png)

次のRFパルスで倒せるのは、その時点までに回復した縦磁化だけです。そのため、信号の大きさはTRの時点での縦磁化の大きさで決まります。

- TRを短く（500ms）すると、脂肪は85%まで回復しているのに対し、脳脊髄液は12%しか回復していません。この差がそのまま明るさの差になり、脂肪が明るく、水が暗く写ります。これがT1強調画像です
- TRを長く（4000ms）すると、脂肪は100%、脳脊髄液も63%まで回復するので、T1による差は小さくなります

T1強調画像ではTEも短くします。TEが長いと、信号を読むまでの間にT2緩和による差が加わってしまうからです。

T1強調画像で明るく写るのは、T1の短いものです。

- 脂肪（皮下脂肪、骨髄の脂肪など）
- 造影剤（ガドリニウム製剤。1-6）が集まった場所。ガドリニウムは周囲の水のT1を短くするので、血流の多い組織や、血液脳関門が壊れた腫瘍・炎症の部位が明るくなります
- 亜急性期の出血（血液の分解産物のメトヘモグロビン）、たんぱく質を多く含む液体、メラニン

暗く写るのは、T1の長い水（脳脊髄液、尿、胆汁）や、水を多く含む病変（浮腫、多くの腫瘍）です。空気や皮質骨からは信号がほとんど得られないので、T1強調画像に限らず暗く写ります。

T1強調画像は解剖の形を見やすく、造影検査の基本にもなります。造影前と造影後のT1強調画像を比べると、造影剤で明るくなった部分が分かります。

出典: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

##### T1強調画像で見る病気の例

![Tectal lipoma on T1-weighted MRI before and after contrast](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_t1_lipoma.jpg)

別の目的の検査で偶然見つかった、中脳の背側（四丘体板）の脂肪腫です。左の造影前のT1強調画像（冠状断）で、脂肪でできた腫瘍が周りの脳よりはっきり明るく写っています。中央（冠状断）と右（矢状断）は造影剤を使ったあとのT1強調画像で、腫瘍の明るさはほとんど変わっていません。T1の短い脂肪は、造影剤がなくても明るく写ります。

出典: Wikimedia Commons — [Tectales Lipom 69jw - MRT T1 nativ und KM coronar T1 KM sagittal - 001.jpg](https://commons.wikimedia.org/wiki/File:Tectales_Lipom_69jw_-_MRT_T1_nativ_und_KM_coronar_T1_KM_sagittal_-_001.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0)

![Subdural hematoma on T1-weighted MRI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_t1_subdural_hematoma.jpg)

硬膜下血腫（脳の表面と、脳を包む硬膜のあいだにたまった血液）の、造影前のT1強調画像（冠状断）です。画像の左側（患者さんの右側）で、脳の表面に沿って三日月形の明るい帯が見えます。出血から数日〜数週たった血腫では、ヘモグロビンが分解してできたメトヘモグロビンがT1を短くするので、T1強調画像で明るく写ります。

出典: Wikimedia Commons — [Subdurales Haematom MRT T1 nativ cor.jpg](https://commons.wikimedia.org/wiki/File:Subdurales_Haematom_MRT_T1_nativ_cor.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0)

#### 横磁化の減衰：T2緩和のグラフ

90°パルスで横に倒した直後を時刻0として、横磁化 $M_{xy}$（磁場に垂直な成分）が減衰する様子を示します。横軸はT2を単位にした時間、縦軸は最大の横磁化 $M_0$ で割った値です。減衰は $M_{xy}(t)/M_0 = e^{-t/T_2}$ に従い、時刻T2で37%、T2の2倍で14%、3倍で5%まで下がります。グラディエントエコー法ではT2の代わりにT2*で減衰するので、曲線はこれより速く下がります。

![T2 decay curve of transverse magnetization](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/t2_decay_mxy.png)

出典: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`。`uv run python instructor/make_figures_1_2_mri.py` で作り直せます）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### T2強調画像：TEを長くして、横磁化の減衰の差を写す

T2強調画像では、まずTRを長くして、T1による差をできるだけ小さくします。T1の差が残っていると、T1の短い組織を明るくする効果と、T2の長い組織を明るくする効果が重なります。水はT1もT2も長いので2つの効果が逆向きに働き、T2の違いが明るさに表れにくくなります。次の図は、TR 4000msのあとに倒した横磁化が、脂肪（T2 約80ms）と脳脊髄液（T2 約2000ms）でどう減衰するかを示します。出発点の高さが違うのは、4000msの間に脳脊髄液の縦磁化が63%までしか回復しないからです（T1強調画像の図の右側の点）。

![Transverse decay of fat and CSF with short and long TE](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/t2_contrast_te.png)

- TEを短く（15ms）すると、どちらもほとんど減衰していないので、明るさの差は出発点の高さで決まります。この条件では脂肪（0.83）のほうが脳脊髄液（0.63）より明るくなります
- TEを長く（100ms）すると、T2の短い脂肪は0.29まで減衰し、T2の長い脳脊髄液は0.60を保ちます。水が明るく写るT2強調画像になります

2本の曲線はTE 40ms付近で交わります。交点より短いTEではT2の差が明るさに十分反映されないので、T2強調画像ではTEを交点より十分長くとります。

ここまでの2つの図は、1つの式にまとめられます。単純化したスピンエコー法では、画素の信号の強さ $S$ はおおよそ

$$S \propto \rho\,\left(1 - e^{-\mathrm{TR}/T_1}\right)\,e^{-\mathrm{TE}/T_2}$$

で表せます。$\rho$ は水素原子核の密度（プロトン密度）です。括弧の部分がTRで決まるT1の効果、最後の項がTEで決まるT2の効果です。TRを長く、TEを短くすると両方の効果が小さくなり、$\rho$ の差が主に残ります。これがプロトン密度（PD）強調画像です。

この式で計算すると、TE 100msの脂肪は脳脊髄液の半分以下の明るさになります。ただし、臨床で使う高速スピンエコー法のT2強調画像では、脂肪はこの計算より明るく写ります。表で脂肪を「明るい〜中間（撮像法による）」としたのはこのためで、脂肪の信号を抑える脂肪抑制を組み合わせることもよくあります。

T2強調画像で明るく写るのは、T2の長い水を多く含むものです。

- 脳脊髄液、尿、胆汁、関節液、嚢胞の内容
- 浮腫、炎症、多くの腫瘍。病変の多くは正常な組織より水を多く含むので、T2強調画像で明るくなります

暗く写るのは、T2の短いものと、信号を出さないものです。

- 皮質骨、空気、石灰化
- 古い出血に残るヘモジデリン
- 速く流れる血液。信号を読む前に撮影している断面を通り抜けてしまうので、血管の内腔が黒く抜けます（flow void）

T2強調画像は、病変を見つけるための基本の画像です。ただし脳の表面や脳室に接する病変は、隣の脳脊髄液も明るいので境界が分かりにくくなります。この場合は、脳脊髄液の信号だけを抑えたFLAIR（fluid-attenuated inversion recovery）画像を使います。

出典: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 頭部で見るT1強調・T2強調・PD強調画像

撮影条件を変えたときの見かけの違いを示した3枚です。左がT1強調（横断面）、中央がT2強調、右がPD強調（中央と右は冠状面）で、断面の向きは揃っていません。

- 脳室の脳脊髄液は、T1強調で暗く、T2強調で明るく写っています
- 脳の白質（神経線維の束）は、T1が灰白質より短いので、T1強調では灰白質より明るく写ります。T2強調では明るさが逆になり、灰白質のほうが明るく写ります
- PD強調では、組織ごとの明るさの差がT1強調やT2強調より小さくなっています

![T1-weighted, T2-weighted and PD-weighted MRI of the brain](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/t1_t2_pd.jpg)

出典: Wikimedia Commons — [T1t2PD.jpg](https://commons.wikimedia.org/wiki/File:T1t2PD.jpg) ／ 画像: KieranMaher（Heggie, Liddell & Maher (2000) の資料を改変） ／ ライセンス: パブリックドメイン（Public domain）

#### T2強調画像で見る病気の例

![Multiple sclerosis lesions on T2-weighted MRI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_t2_ms.png)

多発性硬化症（脳や脊髄の神経線維を覆う髄鞘が、免疫の異常によって繰り返し壊される病気）のT2強調画像（横断面）です。大脳の白質に、周りより明るい小さな斑点（病変）が左右に散らばっています。髄鞘が壊れた部分では水が増えてT2が長くなるので、明るく写ります。脳室の脳脊髄液も明るく写っています。

出典: Wikimedia Commons — [MSMRI.png](https://commons.wikimedia.org/wiki/File:MSMRI.png) ／ 作者: James Heilman, MD ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0)

![Lumbar disc protrusion on T2-weighted MRI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_t2_disc.jpg)

腰椎のT2強調画像（矢状断。画像の左が体の前、右が背中）です。白い楕円で囲んだ第5腰椎の高さの椎間板が、背中の側にわずかに突き出しています（椎間板の突出）。元の画像の説明では、左側の神経根を圧迫しているとされています。T2強調画像では脊柱管の中の脳脊髄液が明るく写るので、神経の通り道がどれだけ狭くなったかを確かめやすくなります。

出典: Wikimedia Commons — [Spinal-disc-protrusion-l5.jpg](https://commons.wikimedia.org/wiki/File:Spinal-disc-protrusion-l5.jpg) ／ 作者: Damato ／ ライセンス: パブリックドメイン（Public domain）

### 1-4. パルスシーケンス：スピンエコー法、グラディエントエコー法、SE-EPI

RFパルスと3方向の傾斜磁場をどの順番とタイミングで使い、いつ信号を読み取るかの手順を、パルスシーケンスと呼びます。1-3のT1強調画像やT2強調画像も、1-2の位置のエンコードも、パルスシーケンスの中で実現されます。基本になるパルスシーケンスは、信号のピーク（エコー）の作り方によって、スピンエコー法とグラディエントエコー法の2つに分けられます。

#### エコーを作る2つの方法

1-3で見たとおり、RFパルスで倒した横磁化は、スピンごとの回転の速さのわずかな違いで位相がばらばらになり、信号が小さくなっていきます。回転の速さの違いには、組織の中の分子の動きによるもの（T2の原因）のほかに、装置の磁場の不均一や、物質ごとの磁化されやすさ（磁化率）の違いによるもの、そして位置のエンコードのためにかけた傾斜磁場によるものがあります。エコーは、ばらばらになった位相をもう一度そろえ直して作ります。

![Echo formation in spin echo and gradient echo](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_echo_formation.png)

- **(a) スピンエコー**：90°パルスから TE/2 たったところで180°パルスを当てると、各スピンの位相の符号が反転します（φ → −φ）。スピンは同じ速さで回り続けるので、先に進んでいたスピンは後ろから、遅れていたスピンは前から同じ時間をかけて戻り、TEでちょうど位相がそろいます。徒競走で、速さの違う走者が合図で一斉に折り返すと、同時にスタートラインに戻ってくるのと同じです。磁場の不均一によるずれも打ち消されるので、TEでの信号は、T2による減衰だけで決まります（図の下段で、TEの信号がT2の減衰の線（0.64）まで戻っています）
- **(b) グラディエントエコー**：180°パルスを使わず、読み取りの傾斜磁場をいったん負にかけてから正に反転させます。傾斜磁場によるずれはTEで打ち消されますが、磁場の不均一や磁化率によるずれは残ります。そのため、TEでの信号はT2*で減衰し、スピンエコーより小さくなります（図では0.20）

#### スピンエコー法（SE）

![Pulse sequence diagrams of SE, GRE and SE-EPI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_sequence_diagrams.png)

パルスシーケンス図は、上から順にRFパルス、スライス選択、位相エンコード、読み取り（周波数エンコード）の傾斜磁場、受信する信号を、時間の流れに沿って並べたものです。

(a) のスピンエコー法では、スライス選択の傾斜磁場をかけながら90°パルスを当て、位相エンコードの傾斜磁場（段になった線は、TRごとに強さを変えることを表します）をかけてから、TE/2 で180°パルスを当てます。TEで生じるエコーを、読み取りの傾斜磁場をかけながら記録すると、k空間の1行が埋まります。これをTRごとに位相エンコードの強さを変えて繰り返します。

磁場の不均一の影響を受けにくいので、金属や空気の近くでも歪みや信号の欠けが少なく、T1強調画像とT2強調画像の基本になる方法です。ただし、TRごとに1行しか埋まらないので時間がかかります。そのため、実際には1-7の高速スピンエコー法で撮影することがほとんどです。

#### グラディエントエコー法（GRE）

(b) のグラディエントエコー法では、90°より小さい角度 α（フリップ角）だけ倒すRFパルスを使います。縦磁化を一部残したまま倒すので、次のRFパルスまでに縦磁化を回復させる時間が短くて済み、TRを数ミリ秒〜数十ミリ秒まで短くできます。180°パルスがないので、TEも短くできます。

グラディエントエコー法のコントラストは、TR、TE、フリップ角の組み合わせで決まります。TRが短くフリップ角が大きいとT1強調に、フリップ角が小さくTEが長いとT2*強調になります。TRが短く速いので、次のような撮影に使われます。

- 息止めの間に撮る腹部のT1強調画像や、造影剤のダイナミック撮影（1-6）。3次元のデータも短時間で集められます
- TOF血管画像（1-5）。短いTRで背景の組織を飽和させます
- T2*強調画像。スピンエコーでは打ち消されてしまう磁場のずれを信号の低下として捉えるので、古い出血に残るヘモジデリンなど、周りの磁場を乱す物質が暗く写ります
- 心臓の動きを動画で撮るシネMRI

2-1で使う腹部のT1強調画像（FLASH）も、グラディエントエコー法で撮影されています。一方で、磁場の不均一の影響を受けやすいので、金属や空気の近くでは信号が欠けたり歪んだりしやすくなります。

#### SE-EPI（スピンエコー型エコープラナー法）

(c) のSE-EPIは、90°パルスと180°パルスでスピンエコーを作り、そのスピンエコーの前後で、読み取りの傾斜磁場を正と負に高速で切り替え続けます。傾斜磁場を1回反転させるたびにグラディエントエコーが1つでき、そのたびにk空間の1行を記録します。行と行のあいだには、位相エンコードの小さな傾斜磁場（ブリップ）を入れて、次の行に移ります。k空間の中心の行を記録する時刻が、スピンエコーのTEになるように並べます。

![k-space trajectories of SE/GRE and EPI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_kspace_trajectory.png)

SEやGREが1回の励起でk空間の1行だけを埋める（左）のに対し、EPIは1回の励起で、ジグザグにk空間全体を埋めます（右）。1枚の画像を0.1秒程度で撮影できるので、体の動きの影響をほとんど受けません。

- **拡散強調像（1-5）**：180°パルスの前後に置いたMPG（図の破線）で拡散の情報を加え、SE-EPIで読み取るのが標準的な方法です
- **脳の機能画像（fMRI）や灌流画像（1-6のDSC）**：180°パルスを使わないグラディエントエコー型のEPI（GRE-EPI）で、T2*の変化を短い時間間隔で繰り返し捉えます

EPIでは、k空間の隣り合う行を記録する時刻が、1行あたり1ミリ秒弱ずつずれていきます。1-2の周波数エンコードでは1行を数ミリ秒で記録したので、位相方向は周波数方向に比べて、同じ周波数のずれが何十倍もの位置のずれになります。そのため、磁場の不均一による歪みや、脂肪の位置のずれ（化学シフト）が位相方向に大きく出ます。脂肪抑制と、1-7のパラレルイメージングによる読み取りの短縮を組み合わせて使うのが一般的です。

| | スピンエコー法（SE） | グラディエントエコー法（GRE） | SE-EPI |
|---|---|---|---|
| エコーの作り方 | 180°パルス | 読み取りの傾斜磁場の反転 | 180°パルス＋傾斜磁場の反転の繰り返し |
| TEでの減衰 | T2 | T2* | T2（k空間の中心） |
| TR・撮影時間 | TRが長く、遅い | TRが短く、速い | 1回の励起で1枚。最も速い |
| 磁場の不均一の影響 | 小さい | 大きい | 位相方向の歪みが大きい |
| 主な用途 | T1強調、T2強調（高速スピンエコー法として） | 3次元T1強調、ダイナミック造影、TOF、T2*強調、シネ | 拡散強調像 |

出典（この節の図）: 自作（NumPy によるシミュレーションと matplotlib による模式図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-5. T1・T2以外のコントラスト：拡散と血流

明るさの差の元になる物理現象を、ここではコントラストの源と呼びます。1-3のT1強調画像とT2強調画像は、緩和の速さをコントラストの源にしていました。MRIでは、ほかにも水分子の拡散や血液の流れをコントラストの源にできます。それぞれを使うのが、拡散強調像（DWI）とTOF血管画像です。

#### 拡散強調像（DWI）

1つ目のコントラストの源は、水分子の拡散です。拡散強調像（diffusion-weighted imaging、DWI）は、組織の中で水がどれだけ動けるかの違いを、明るさの差として写します。

##### 水分子の拡散

体温の水の中では、水分子が熱によって絶えず動き回り、周りの分子とぶつかるたびに向きを変えています。そのため、1つの分子の道筋は予測のつかないジグザグになります（ブラウン運動）。1つ1つの分子がどこへ行くかは決まりませんが、たくさんの分子について平均をとると、出発点からの広がりは時間とともに決まった規則で大きくなります。ある方向（x方向）の変位を $\Delta x$ とすると、その2乗の平均は

$$\langle \Delta x^2 \rangle = 2 D t$$

になります。$D$ は拡散係数で、その物質の中で分子がどれだけ動きやすいかを表します。体温の自由な水では $D \approx 3.0\times10^{-3}\ \mathrm{mm^2/s}$ です。

DWIで水の動きを観察する時間（拡散時間）は数十msです。50msとすると、1方向の変位の目安 $\sqrt{2Dt}$ は約17µmになります。多くの細胞の大きさは10µm前後なので、水分子はこの時間の間に細胞膜に何度もぶつかります。DWIの画像に細胞の並び方や密度が反映されるのは、このためです。

##### 組織の中で水の動きが妨げられる様子

次の図は、水分子が50msの間にたどる道筋を、コンピュータで模擬したもの（シミュレーション）です。3つの図は同じ出発点（黒い点）と同じ乱数を使っていて、違うのは細胞があるかどうかと、細胞の大きさだけです。灰色の円が細胞で、この模擬では細胞膜は水を通さないものとしています。

![Simulated paths of water molecules in free water, normal tissue and swollen cells](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/dwi_diffusion_paths.png)

- 左（自由水）：障害物がないので、分子は出発点から20µm以上離れた場所まで広がります
- 中央（正常な組織）：細胞の外の水（青）は、細胞のすき間を縫って進むので遠回りになり、広がりが小さくなります。細胞の中の水（橙）は、細胞膜に囲まれた範囲から出られません
- 右（細胞が膨らんだ組織）：急性脳梗塞の細胞性浮腫のように、細胞が膨らんで細胞外のすき間が狭くなった状態です。細胞の外の水は狭いすき間に閉じ込められ、中央の図よりさらに動けなくなります。水全体のうち細胞の中にある割合も増えます

組織が水の動きを妨げるしくみは、2つに分けられます。細胞の外の水は、障害物を避けて遠回りするので広がりにくくなります（hindered diffusion）。細胞の中の水は、動ける範囲そのものが膜で区切られています（restricted diffusion）。臨床でいう「拡散制限」は、この2つを区別せず、水の動きが正常な組織より小さい状態を指すのが一般的です。

##### 変位の分布

分子を20000個に増やし、50ms後のx方向の変位を集計したのが次の図です。青が細胞の外、橙が細胞の中にある分子で、2つを積み上げて表示しています。

![Distribution of displacement along x after 50 ms](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/dwi_displacement_hist.png)

- 自由水では、変位が左右に大きく広がります。変位の標準偏差（SD）は17.2µmで、上の式から求めた約17µmとほぼ一致します
- 正常な組織では、変位の標準偏差が8.6µmと自由水の半分になります。細胞の中の分子（橙）は細胞の大きさより先へ動けないので、0の近くに集まります
- 細胞が膨らんだ組織では、細胞の外の分子（青）の広がりも小さくなり、変位の標準偏差は6.0µmまで下がります

図の右上に書いた信号の残り $S(b=1000)/S(0)$ とADCは、この変位の分布から計算した値です。これらの意味は、このあとMPGのしくみとb値を説明してから確かめます。

この模擬は、細胞を同じ大きさの円に、細胞膜を水を通さない壁に単純化したものです。実際の細胞膜は水をゆっくり通し、細胞の形や大きさもさまざまなので、図の数値は実際の組織の値とは一致しません。図から読み取れるのは、細胞外のすき間が狭くなると水の動ける距離が短くなる、という傾向です。

出典: 自作（NumPyによるシミュレーションとmatplotlibによる作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

##### 動いた水の信号が減るしくみ

DWIでは、スピンエコー法の180°パルスの前と後に、同じ強さの傾斜磁場を1回ずつ短く加えます。この傾斜磁場をMPG（motion probing gradient）と呼びます。

1. 1回目のMPGをかけている間は、場所によって磁場の強さが違うので、スピンの位相の進み方が位置によって変わります（1-2の位相エンコードと同じしくみです）
2. 180°パルスで、それまでの位相の進みが反転します
3. 2回目のMPGで、1回目と同じだけ位相が進みます。その間に動かなかったスピンでは、反転した1回目の進みがちょうど打ち消され、位相が元にそろいます
4. 1回目と2回目の間に動いたスピンは、1回目とは違う場所で2回目のMPGを受けるので、打ち消しきれない位相のずれが残ります

拡散による動きは向きも距離もばらばらなので、残る位相のずれもスピンごとにばらばらになります。位相のばらばらなスピンは互いに打ち消し合うので、信号が小さくなります。水がよく動く場所ほど信号が大きく下がり、動きが制限された場所ほど信号が残って明るく写ります。

MPGで捉えられるのは、MPGをかけた方向の動きだけです。そのため、通常は3方向でそれぞれ撮影して合成します。方向ごとの違いをさらに詳しく調べると、神経線維の走行を推定する拡散テンソル画像（DTI）になります。

DWIは、1回の励起で画像全体のデータを集めるエコープラナー法（EPI）で撮影するのが一般的です。1枚を短時間で撮影できるので、体の動きの影響を受けにくくなります。そのかわり、磁場の不均一で画像が歪みやすくなります（理由は1-4のSE-EPIの項で説明しました）。

##### b値とADC

MPGで動きにどれだけ敏感にするかを表すのがb値（b value、単位 s/mm²）です。MPGが強いほど、長いほど、2回の間隔が長いほど、b値は大きくなります。b値0はMPGをかけない画像で、T2強調画像に近いコントラストになります。頭部ではb値1000 s/mm²程度がよく使われます。

変位の分布の図に書いた $S(b=1000)/S(0)$ は、模擬した分子のそれぞれに変位に比例した位相のずれが残るとして、b値1000 s/mm²のときの全分子の信号を足し合わせた値です。変位の分布が広い自由水では信号が0.05しか残らないのに対し、正常な組織では0.62、細胞が膨らんだ組織では0.76が残ります。変位の分布が狭いほど位相のばらつきも小さく、信号が多く残ります。

信号は、b値が大きくなるにつれて

$$S(b) = S(0)\,e^{-b \cdot \mathrm{ADC}}$$

のように指数関数的に減ります。ADC（apparent diffusion coefficient、見かけの拡散係数）は、その場所で水がどれだけ動きやすいかを表す値です。水の動きを妨げるものがない自由水では、ADCは拡散係数 $D$ に一致します。組織の中では、細胞膜などの障害物のために水の本来の動きやすさより小さな値になるので、「見かけの」と付きます。b値の違う2枚以上の画像から、画素ごとにADCを計算して並べた画像をADC mapと呼びます。ADC mapでは、水がよく動く場所ほど明るく写ります。

変位の分布の図のADCも、$S(b=1000)/S(0)$ をこの式に当てはめて求めました。自由水では $2.96\times10^{-3}\ \mathrm{mm^2/s}$ と、設定した拡散係数にほぼ一致します。正常な組織では $0.48\times10^{-3}$、細胞が膨らんだ組織では $0.27\times10^{-3}$ で、細胞外のすき間が狭くなるとADCが下がることが分かります。なお、組織では、変位の分散から $\langle \Delta x^2 \rangle = 2Dt$ で逆算した値（正常な組織で約 $0.75\times10^{-3}$）と、信号から求めたADCが一致しません。細胞の中の水の変位が正規分布に従わないためです。臨床で使うADCは、信号から求めた値のほうです。

##### 急性脳梗塞がDWIで明るく写る理由

脳の血流が途絶えると、細胞がエネルギー（ATP）を作れなくなり、細胞膜のナトリウムポンプ（Na⁺/K⁺-ATPase）が止まります。細胞の中にナトリウムがたまり、水が細胞内へ移動して細胞が膨らみます（細胞性浮腫）。細胞外のすき間が狭くなるので、水が動ける距離が短くなり、ADCが下がってDWIで高信号になります（模擬の図では右側の状態です）。この変化は発症から早い時期に現れるので、CTで変化が見えない段階でも、DWIで梗塞を捉えられることが多くあります。

細胞が密に詰まった腫瘍（悪性リンパ腫など）や、粘り気の強い膿がたまった膿瘍でも、水の動きが制限されてDWIで高信号になります。

##### DWIとADC mapを一緒に見る理由

DWIの信号は、b値0の画像（T2強調に近い画像）の信号から減っていく形で決まります。そのため、T2の長い組織は、水が十分に動けていてもDWIで明るさが残ることがあります。これをT2 shine-throughと呼びます。本当に拡散が制限されているかは、T2の影響を含まないADC mapで確かめます。DWIで高信号かつADC mapで低信号なら、水の拡散が制限されていると判断します。

![DWI, ADC map and TOF angiography of a cerebral infarction](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/dwi_adc_tof_infarct.jpg)

左前大脳動脈の閉塞による急性脳梗塞の例です。左がDWIで、梗塞部が高信号に写っています。中央はADC mapで、同じ部位が低信号です。DWIで高くADCで低いという組み合わせが、水の拡散が制限されていることを示します。ADC mapでは、自由に動ける脳室や脳溝の脳脊髄液が明るく写っていることも確かめられます。右はTOF血管画像（次に説明します）で、矢印の箇所で血管の信号が失われています。

出典: Wikimedia Commons — [Unilateraler Anteriorinfarkt 70M - MR DWI ADC TOF - 001.jpg](https://commons.wikimedia.org/wiki/File:Unilateraler_Anteriorinfarkt_70M_-_MR_DWI_ADC_TOF_-_001.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0)

##### ほかの例：中脳のごく小さな梗塞

![Tiny acute infarct in the midbrain on DWI and ADC map](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_dwi_midbrain_infarct.jpg)

右の動眼神経（眼を動かす神経）の麻痺が急に起きた患者さんの、中脳の高さのDWI（左、b値1000 s/mm²）とADC map（右）です。赤い円の中の、動眼神経の核にあたるごく小さな領域が、DWIで明るく、ADC mapで暗く写っています。このような小さな新しい梗塞も、DWIとADC mapを組み合わせると見つけられることがあります。

出典: Wikimedia Commons — [Frische Ischaemie im Nucleus nervi oculomotorii rechts - MRT DWI ADC.jpg](https://commons.wikimedia.org/wiki/File:Frische_Ischaemie_im_Nucleus_nervi_oculomotorii_rechts_-_MRT_DWI_ADC.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0)

#### TOF血管画像（time-of-flight angiography）

2つ目のコントラストの源は、血液の流れです。TOF血管画像は、造影剤を使わずに血管（主に動脈）を描く撮像法です。

##### 止まっている組織を暗くし、流れ込む血液を明るくする

TOFでは、グラディエントエコー法で、同じ撮影範囲にRFパルスを短い間隔（TRが数十ms）で繰り返し当てます。T1強調画像の図で見たとおり、TRが短いと縦磁化は次のパルスまでに回復しきれません。撮影範囲にとどまっている組織は、回復の途中で何度も倒し直されるので、縦磁化が小さいままになり、信号が弱くなります。これを飽和と呼びます。

一方、撮影範囲の外から流れ込んでくる血液は、まだRFパルスを受けていないので、縦磁化が十分に回復した状態で入ってきます。この血液は大きな信号を出すので、暗い背景の中で血管だけが明るく写ります。これを流入効果（flow-related enhancement）と呼びます。

グラディエントエコー法を使うのは、スピンエコー法よりTRを短くでき、背景の組織を強く飽和させられるからです。TEも数msと短くして、流れによる位相のずれで信号が落ちるのを抑えます。

##### 明るく写る血管と写りにくい血管

流入効果は、撮影範囲の中の血液が新しい血液と次々に入れ替わるほど強く出ます。

- 撮影範囲を垂直に横切って、速く流れ込む血管は明るく写ります
- 流れの遅い血管や、撮影範囲の中を長く走る血管では、血液が範囲内にとどまる間にパルスを何度も受けて飽和するので、流れの先へ行くほど暗くなります

撮影範囲には動脈血も静脈血も流れ込みます。頭部では、動脈血は主に下から、静脈血は主に上から撮影範囲に入ります。そこで、撮影範囲のすぐ上にだけRFパルスを当てて静脈血を先に飽和させておくと（飽和パルス）、静脈の信号が消えて動脈だけが残ります。

##### 3次元のデータを最大値投影（MIP）で1枚にまとめる

TOFでは3次元のデータを撮影し、最大値投影（MIP、maximum intensity projection）で表示することがよくあります。MIPは、決めた方向に沿って並ぶ画素のうち最も明るい値だけを選び、2次元の画像を作る方法です。NumPyで書けば、3次元配列 `volume` に対する `volume.max(axis=0)` がMIPにあたります。背景が暗く血管が明るいので、MIPで血管の枝分かれを1枚で見渡せます。投影の方向を少しずつ変えた画像を並べると、血管の立体的な走行も分かります。

ただし、MIPでは奥行きの情報が失われるので、重なった血管の前後関係は分かりません。細い血管が背景に埋もれることもあるので、診断では元の断面の画像も確かめます。

##### 読むときの注意

- T1の短い物質（亜急性期の血腫、脂肪）は、短いTRでも縦磁化がよく回復するので、血流がなくても明るく写り、血管と紛らわしくなることがあります
- 狭窄の先では流れが乱れ、スピンの位相がばらばらになって信号が落ちます。このため、狭窄が実際より強く見えることがあります

TOF血管画像は、脳動脈瘤の検出や、動脈の狭窄・閉塞の評価に使われます。

![MIP of a TOF MR angiography showing the circle of Willis](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/tof_mra_mip.jpg)

TOF血管画像のMIPの例です。脳底のウィリス動脈輪と、そこから枝分かれする血管が明るく写っています。前の図の右側も同じ方法で作った画像で、閉塞した前大脳動脈の先が写らなくなっています。

出典: Wikimedia Commons — [Mra-mip.jpg](https://commons.wikimedia.org/wiki/File:Mra-mip.jpg) ／ 作者: SBarnes ／ ライセンス: [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0)

##### 病気の例：中大脳動脈の狭窄

![TOF MR angiography of middle cerebral artery stenosis](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_tof_mca_stenosis.png)

右の中大脳動脈に狭窄のある患者さんのTOF血管画像（MIP）です。右側の画像の矢印の付近で、中大脳動脈が細く写っています。上の「読むときの注意」のとおり、狭窄の先では流れが乱れて信号が落ちるので、実際より強い狭窄に見えることがある点に注意します。

出典: Wikimedia Commons — [TOF MRI angiography of right middle cerebral artery stenosis.png](https://commons.wikimedia.org/wiki/File:TOF_MRI_angiography_of_right_middle_cerebral_artery_stenosis.png) ／ 作者: Shazia Mirza, Sankalp Gokhale ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 4つのコントラスト形成機構の比較

| コントラスト | 源になる物理現象 | 測っている量 | 撮影条件の要点 | 例 |
|---|---|---|---|---|
| T1強調 | スピンから格子へのエネルギー移動（T1緩和） | TRの時点での縦磁化の回復の程度 | 短いTR、短いTE | 脂肪や造影剤が集まった部位が明るく、脳脊髄液が暗い |
| T2強調 | 局所磁場の違いによる位相のばらつき（T2緩和） | TEの時点で残っている横磁化 | 長いTR、長いTE | 脳脊髄液や浮腫などの病変が明るい |
| 拡散強調像（DWI） | 水分子の拡散 | 水がどれだけ動けるか（ADC） | MPGを加える（b値） | 急性脳梗塞がDWIで高信号、ADC mapで低信号 |
| TOF血管画像 | 血流による流入効果 | 飽和していない血液の流入 | 短いTRで背景を飽和させる | 造影剤なしで動脈を描く |

### 1-6. MRI造影剤

CTのヨード造影剤は、ヨードそのものがX線を弱めることで写ります。MRIの造影剤は、造影剤そのものが写るのではありません。造影剤は周りの水の水素原子核の緩和時間を短くし、その結果として水の信号の強さが変わります。T1を短くして明るくする造影剤を陽性造影剤、T2やT2*を短くして暗くする造影剤を陰性造影剤と呼ぶことがあります。

#### ガドリニウム造影剤

最もよく使われるのは、ガドリニウム（Gd）を含む造影剤です。ガドリニウムのイオン（Gd³⁺）は対になっていない電子を7個もち、非常に強い常磁性を示します。電子の磁気モーメントは水素原子核の約660倍もあるので、Gd³⁺の近くに来た水分子は、強く揺らぐ局所磁場を受けます。1-3で見たとおり、T1緩和は局所磁場の揺らぎで起こるので、Gd³⁺の周りの水はT1が大きく短くなります。

T1の短くなり方は、造影剤の濃度 $C$ に比例します。

$$\frac{1}{T_1} = \frac{1}{T_{1,0}} + r_1 C$$

$T_{1,0}$ は造影剤がないときのT1、$r_1$ は緩和能と呼ばれる造影剤ごとの定数で、一般的な製剤では1.5テスラで3〜5 /(mM·s)程度です。T2についても同じ形の式が成り立ちます。

![T1 and T2 shortening and signal change with gadolinium](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/gd_relaxivity.png)

T1 = 1000 ms、T2 = 100 msの組織に、緩和能 $r_1$ = 4、$r_2$ = 5 /(mM·s)の造影剤が入ったときの計算です。

- (a) 0.5 mMでT1は1000 msから335 msまで短くなりますが、T2は100 msから80 msにしか変わりません。低い濃度では、主にT1が短くなります
- (b) 1-3のスピンエコーの式で信号を計算すると、T1強調画像（TR 500 ms、TE 15 ms）では、0.5 mMで信号が約1.9倍になります。T2強調画像ではわずかに暗くなります。ただし、濃度が高くなりすぎると（この条件では1.4 mMより上）、T2の短縮の効果が勝って、T1強調画像でもかえって暗くなります。尿がたまった膀胱などで、造影剤が濃い部分が暗く写ることがあるのはこのためです

そのままのGd³⁺は体内で毒性をもつので、キレート剤という分子で包み込み、体内で離れないようにしてあります。キレートの形には鎖状（線状型）と環状（マクロ環型）があり、環状のほうがGd³⁺が外れにくいとされています。通常の使用量は体重1 kgあたり0.1 mmolで、静脈から入れたあとは、ヨード造影剤と同じように細胞外液に広がり、腎臓から尿に排泄されます。

ガドリニウム造影剤の主な使い方は、次のとおりです。

- **造影T1強調画像**：血液脳関門が壊れた脳腫瘍や転移、炎症、多発性硬化症の活動性の病変などが明るく写ります（CTの造影と同じく、正常な脳の組織には漏れ出しません）
- **ダイナミック造影**：造影剤を入れながら同じ部位を繰り返し撮影し、明るくなる速さと抜け方から、肝臓・乳房・前立腺などの腫瘍の性質を調べます
- **造影MR血管撮影**：T1が短くなった血液を明るく描き、血管の形を調べます
- **灌流画像（DSC）**：造影剤が一塊になって脳を初めて通過する間は、血管の中の濃度が高く、周りの磁場を乱します。T2*強調画像（1-3で見たT2*の減衰を強く反映する画像）で信号の一時的な低下を追い、脳の血流を調べます
- **遅延造影**：造影剤を入れて10分ほどたってから撮影すると、心筋梗塞のあとの瘢痕や線維化した心筋に造影剤が残って明るく写ります

安全上の注意として、腎機能が大きく低下した患者さんでは、皮膚や内臓が硬くなる腎性全身性線維症（NSF）が報告されており、主に線状型の造影剤で起こりました。また、線状型の造影剤を繰り返し使った人では、脳の歯状核や淡蒼球にガドリニウムがわずかに残り、T1強調画像で明るく写ることが報告されています。このため欧州では2017年に一部の線状型の造影剤の使用が制限され、現在は環状型の造影剤を、必要な量だけ使うのが基本になっています。

出典: 自作（matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### ガドリニウム造影剤を使った画像の例

![Brain metastases on T1-weighted MRI before and after gadolinium](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_gd_metastases.jpg)

肺がんの多数の脳転移です。左は造影前、右は造影後のT1強調画像（同じ高さの横断面）です。造影前にはほとんど分からない小さな転移が、造影後には白い点として多数写っています。転移の部分では血液脳関門が壊れているので、ガドリニウムが血管の外へ漏れ出てT1が短くなり、明るく写ります。

出典: Wikimedia Commons — [BC - Hirnmetastasen MRT T1 ax.jpg](https://commons.wikimedia.org/wiki/File:BC_-_Hirnmetastasen_MRT_T1_ax.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0)、[BC - Hirnmetastasen MRT T1KM ax.jpg](https://commons.wikimedia.org/wiki/File:BC_-_Hirnmetastasen_MRT_T1KM_ax.jpg)（同じ作者・ライセンス）。本教材で2枚を横に並べ、見出しの文字を加えた

![Brain abscess with ring enhancement](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_gd_abscess.jpg)

脳膿瘍（脳の中に膿がたまった状態）の造影後のT1強調画像です。膿を囲む壁が輪状に明るく造影され、その周りの脳は、むくみ（浮腫）のためにやや暗く写っています。輪状に造影される腫瘍と見分けるには、1-5のDWIが役に立ちます。粘り気の強い膿は水の拡散が制限されるので、DWIで明るく写ります。

出典: Wikimedia Commons — [Brain abscess - MRI T1 KM axial.jpg](https://commons.wikimedia.org/wiki/File:Brain_abscess_-_MRI_T1_KM_axial.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0)

![Liver hemangioma on T2-weighted, T1-weighted and dynamic contrast-enhanced MRI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_gd_liver_hemangioma.jpg)

肝血管腫（血管が集まってできた良性の腫瘍）の例です。(A) T2強調画像で明るく、(B) 造影前のT1強調画像で暗く写っています。造影剤を入れると、(C) 動脈相では腫瘍の縁から染まり始め、(D) 遅い時期には腫瘍全体が明るくなります。ダイナミック造影では、このような染まり方の時間変化から、腫瘍の種類を見分けます。

出典: Wikimedia Commons — [Angioma epatico-RM.jpg](https://commons.wikimedia.org/wiki/File:Angioma_epatico-RM.jpg) ／ 作者: Nils Albiin（Albiin N. MRI of Focal Liver Lesions. 2012. [PMC3462338](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3462338/) の図） ／ ライセンス: [CC BY 2.5](https://creativecommons.org/licenses/by/2.5)

#### 肝細胞特異性造影剤（ガドキセト酸）

ガドキセト酸（Gd-EOB-DTPA）は、ガドリニウム造影剤の一種で、日本では肝臓の検査に広く使われています。静脈から入れた直後は、ほかのガドリニウム造影剤と同じように血管と細胞外液に広がるので、動脈相や門脈相のダイナミック造影ができます。そのあと、正常な肝細胞が細胞膜の輸送体（OATP1B1/1B3）を通して取り込み、胆汁に排泄します（量の約半分。残りは尿へ排泄）。

注入から約20分後の肝細胞相では、造影剤を取り込んだ正常な肝臓がT1強調画像で明るく写ります。働いている肝細胞をもたない病変（多くの肝細胞がん、転移、嚢胞など）は造影剤を取り込まないので、周りより暗く抜けて見えます。このため、小さな病変を見つけやすくなります。胆管も造影剤を含む胆汁で明るく写ります。緩和能が高いので、使う量は通常のガドリニウム造影剤の4分の1（体重1 kgあたり0.025 mmol）です。


![Focal nodular hyperplasia and hepatocellular adenoma with a hepatocyte-specific contrast agent](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_eob_fnh_adenoma.jpg)

限局性結節性過形成（FNH、矢頭）と肝細胞腺腫（矢印）という2種類の良性の腫瘍がある患者さんに、肝細胞特異性造影剤を使った例です。

- (A) T2強調画像、(B) ADC map、(C) 造影前のT1強調画像：どちらの病変も、周りの肝臓との明るさの差は小さくなっています
- (D) 動脈相：どちらも明るく染まります
- (E) 門脈相：どちらも周りと同じくらいの明るさになります
- (F) 肝細胞相：働いている肝細胞を含むFNHは周りより明るく、腺腫は周りよりやや暗く写ります

ダイナミック造影の(D)(E)では区別しにくい2つの病変が、肝細胞相で見分けられます。

出典: Wikimedia Commons — [CMIR-8-107 F5.jpg](https://commons.wikimedia.org/wiki/File:CMIR-8-107_F5.jpg) ／ 作者: Nils Albiin（Albiin N. MRI of Focal Liver Lesions. 2012. [PMC3462338](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3462338/) の図） ／ ライセンス: [CC BY 2.5](https://creativecommons.org/licenses/by/2.5)


多くの肝細胞がん（HCC）や転移などの悪性腫瘍は、働いている肝細胞をもたない（あるいは取り込みの輸送体が減っている）ので、肝細胞相では造影剤を取り込まず、明るくなった周りの肝臓の中で暗く抜けて写ります。次の2つは、その例です。

![Neuroendocrine liver metastasis on gadoxetic acid-enhanced MRI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_eob_metastasis.jpg)

神経内分泌腫瘍の肝転移（矢印）です。

- (a) T2強調画像：境界のはっきりした、とても明るい結節として写っています
- (b) 造影前の脂肪抑制T1強調画像：周りより暗く写っています
- (c) 動脈相：結節全体が強く染まります
- (d) 門脈相、(e) 平衡相：造影剤が速く抜け、平衡相でよりはっきり暗くなります
- (f) 肝細胞相（Gd-EOB-DTPA投与の10分後）：周りの肝臓は明るくなりますが、結節は造影剤を取り込まず、暗く写ります

出典: Grazioli L, Olivetti L, Mazza G, Bondioni MP. MR Imaging of Hepatocellular Adenomas and Differential Diagnosis Dilemma. *Int J Hepatol*. 2013;2013:374170. [doi:10.1155/2013/374170](https://doi.org/10.1155/2013/374170)（[PMC3623472](https://pmc.ncbi.nlm.nih.gov/articles/PMC3623472/)）の Figure 13 ／ ライセンス: [CC BY 3.0](https://creativecommons.org/licenses/by/3.0/) ／ 本教材で、縦に並んだ6枚を2段×3列に並べ替え、(a)〜(f) のラベルを付け直した

![Hepatocellular carcinoma on dynamic and late-phase liver MRI](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_case_eob_hcc.jpg)

肝右葉（画像の左側）の大きな肝細胞がんです。左から、脂肪抑制T2強調画像、造影後の早い時期（動脈相）、3分後、20分後のT1強調画像です。20分後の画像では、周りの肝臓が明るくなっているのに対し、腫瘍は暗く抜けています。元の画像の説明に造影剤の名前は書かれていませんが、20分後に肝臓全体が明るく写っていることから、肝細胞特異性造影剤を使った肝細胞相の画像と考えられます。

出典: Wikimedia Commons — [Fetthaltiges HCC 81M - CT und MRT - 001.jpg](https://commons.wikimedia.org/wiki/File:Fetthaltiges_HCC_81M_-_CT_und_MRT_-_001.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0) ／ 本教材で、8枚組のうちMRIの4枚（下段）を切り出し、撮像法のラベルを加えた（加工した画像も CC BY-SA 4.0 で提供）

ただし、高分化の肝細胞がんの一部は輸送体を保っていて、肝細胞相で周りと同じか明るく写ることがあります。肝細胞相の明るさだけで良性か悪性かを決めることはできず、T2強調画像、拡散強調像、ダイナミック造影の所見とあわせて判断します。

#### 酸化鉄を使う造影剤とその他の造影剤

- **超常磁性酸化鉄（SPIO）**：酸化鉄のナノ粒子で、静脈から入れると肝臓のクッパー細胞（異物を取り込む細胞）に取り込まれます。粒子は非常に大きな磁化をもち、周りの磁場を乱してT2とT2*を短くするので、正常な肝臓がT2強調画像やT2*強調画像で暗くなります。クッパー細胞をもたない腫瘍は暗くならず、相対的に明るく残ります。代表的な陰性造影剤です
- **経口の消化管造影剤**：MRI胆管膵管撮影（MRCP）は、強いT2強調画像で胆管や膵管の中の水を明るく描く検査です。胃や十二指腸の水も明るく写って重なるので、検査の前にクエン酸鉄アンモニウムや塩化マンガンを含む液を飲み、消化管の中の水のT2を短くして暗くします
- **過分極ガス**：キセノン129のガスを特殊な方法で強く磁化させてから吸入し、ガスそのものの信号で肺の換気を画像にします。米国では2022年に臨床での使用が承認されました。ほかの造影剤と違い、水ではなく造影剤そのものの信号を写します

### 1-7. MRIの高速撮像法

1-2で見たとおり、MRIの撮影時間はおおよそ次の式で決まります。

$$\text{撮影時間} \approx \mathrm{TR} \times \text{位相エンコード数} \times \text{加算回数}$$

撮影時間が長いと、体の動きによるアーチファクトが出やすく、患者さんの負担も大きくなります。高速撮像法は、この式のどこかを小さくする工夫です。1回のTRで複数の行を集める方法（高速スピンエコー法、EPI）と、集める行の数そのものを減らす方法（ハーフフーリエ法、パラレルイメージング、圧縮センシング）に分けられます。

#### 高速スピンエコー法（FSE、TSE）

通常のスピンエコー法では、90°パルスのあとに180°パルスを1回当てて1つのエコーを得て、k空間の1行を埋めます。高速スピンエコー法（fast spin echo、FSE。turbo spin echo、TSEとも呼ぶ）では、1回の90°パルスのあとに180°パルスを続けて何回も当て、エコーを次々に作ります（エコートレイン）。エコーごとに位相エンコードの強さを変えれば、1回のTRで複数の行を埋められます。

![Fast spin echo: echo train and k-space filling](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_fse.png)

- (a) 1回の90°パルスのあとに8回の180°パルスを当て、8つのエコーを得る例です。エコーの大きさは、T2緩和によって後になるほど小さくなります
- (b) 8つのエコーでk空間の行を分担して埋める様子です。エコーの数（エコートレイン長）が8なら、撮影時間は通常のスピンエコー法の8分の1になります

1-2で見たとおり、k空間の中心が画像のコントラストを決めます。そのため、画像のTEは「k空間の中心を埋めたエコーの時間」になり、これを実効TEと呼びます。図では4番目のエコー（48 ms）で中心を埋めているので、実効TEは48 msです。エコートレインを長くするほど速くなりますが、k空間の行ごとに信号の大きさが違うため画像が位相方向にぼけやすくなり、180°パルスが増えるのでSAR（1-1）も上がります。現在のT2強調画像は、ほとんどがこの方法で撮影されています。1-3で触れた「T2強調画像で脂肪が明るく写る」のも、この方法の性質によるものです。

#### ハーフフーリエ法

1-2で見たとおり、画像が実数なら、k空間の原点について対称な2点の値は互いに複素共役になります。つまり、k空間の半分がわかれば、残りの半分は計算で作れます。ハーフフーリエ法（half-Fourier、partial Fourier）は、位相方向のk空間の半分強（たとえば5/8）だけを集め、残りを対称性から埋める方法で、撮影時間をほぼ半分にできます。

![Half-Fourier acquisition and reconstruction](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_partial_fourier.png)

- (a) 位相方向に5/8の行だけを集め、残りの3/8（暗い部分）を集めない例です
- (b) 集めなかった行を0のまま画像に戻すと、細かい縞の成分が片側しかないので、輪郭がぼけて波打ちが出ます
- (c) 集めなかった行を、反対側の値の複素共役で埋めると、元の画像（d）とほぼ同じになります

この模型の画像は完全に実数なので、(c)は元の画像と一致します。実際の画像には磁場の不均一などによるゆるやかな位相が含まれるので、中心付近を半分より少し多めに集めてその位相を見積もり、補正してから対称性を使います。集めるデータが少ない分、SN比は下がります。

#### HASTE法

HASTE（half-Fourier acquisition single-shot turbo spin echo）は、高速スピンエコー法とハーフフーリエ法を組み合わせ、1回の90°パルスのあとのエコートレインだけで、必要なk空間の行をすべて埋める方法（シングルショット）です。1枚を1秒未満で撮影できるので、呼吸や腸の動きがあっても動きのアーチファクトがほとんど出ません。息止めの難しい患者さんの腹部、MRI胆管膵管撮影（MRCP）、胎児のMRIなどに使われます。エコートレインがとても長いので、強いT2強調になり、T2の短い組織はぼけやすくなります。2-1で使う腹部のT2強調画像も、HASTEで撮影されています。

1回の励起のあとに傾斜磁場を高速に切り替えて、グラディエントエコーを次々に作り、1回で画像全体のデータを集めるのがEPI（1-5）です。1枚を0.1秒程度で撮影でき、拡散強調像や脳の機能画像（fMRI）に使われます。

#### パラレルイメージング（SENSE、GRAPPA）

1-1で見たとおり、現在の受信コイルは小さなコイルを多数並べたアレイコイルです。コイルごとに体のどこに近いかが違うので、同じ場所の信号でも、コイルによって受け取る強さ（感度）が違います。パラレルイメージングは、この感度の違いを使って、位相エンコードの行を間引いた分の情報を補う方法です。

![SENSE: unfolding with coil sensitivities](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/mri_sense.png)

- 上段：模型と、上側に置いたコイル1、下側に置いたコイル2の感度の分布です
- 下段の左と中央：位相方向の行を1行おきに間引くと（加速率 R = 2）、1-2の $\Delta k = 1/\mathrm{FOV}$ の関係から撮影範囲が半分になり、はみ出した上下の部分が重なって写ります。重なり方は同じでも、2つのコイルでは上下の部分の明るさの比が違います
- 下段の右：画素ごとに「上の部分の値」と「下の部分の値」の2つを未知数とし、2つのコイルの値と感度から2つの式を立てて解くと、重なりのない画像に戻せます。これがSENSE（sensitivity encoding）です

撮影時間は加速率 R 分の1になります。そのかわり、集めるデータが減る分（$\sqrt{R}$ 倍）と、コイルの配置によって式が解きにくくなる分（gファクター）だけ、SN比が下がります。コイルの感度の分布は、本撮影の前の短い撮影で測っておきます。GRAPPAは同じ考え方を、画像ではなくk空間の上で行う方法で、間引いた行の値を、周りの行とコイルごとのデータから推定します。現在は、多くの撮影で2〜3倍程度の加速がふつうに使われています。

#### 圧縮センシングとディープラーニング

さらに、行を不規則に間引いて集め、「画像は少ない成分で表せる」という性質を使って、逐次近似の計算で画像を作る圧縮センシング（compressed sensing）もあります。最近は、ディープラーニングで少ないデータからノイズの少ない画像を作る方法も臨床で使われています。これらは、1-3のノートブック（画質改善技術）で扱う逐次近似再構成とディープラーニングの考え方と共通です。

| 方法 | 小さくするもの | 主な代償 |
|---|---|---|
| 高速スピンエコー法 | TRあたりの行数を増やす（エコートレイン長の分だけ速い） | 位相方向のぼけ、SARの増加 |
| EPI | 1回の励起で全部の行を集める | 磁場のずれによる歪み |
| ハーフフーリエ法 | 集める行を約半分に | SN比の低下、位相の補正が必要 |
| HASTE | 高速スピンエコー法 + ハーフフーリエ法を1回の励起で | 強いT2強調、ぼけ |
| パラレルイメージング | 集める行を R 分の1に | SN比の低下（$\sqrt{R}$ とgファクター） |
| 圧縮センシング、ディープラーニング | 集める行を減らす | 計算時間、画像のもっともらしさの検証が必要 |

出典（この節の図）: 自作（NumPy によるシミュレーションと matplotlib による作図。生成スクリプト: `instructor/make_figures_1_2_mri.py`）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-8. 静磁場の強さ

MRI装置の静磁場の強さ（1-1）は、単位テスラ（T）で表します。臨床で使われている装置は、おおよそ次のように分けられます。

- **低磁場（0.5 T以下）**：永久磁石などを使い、上下の磁石のあいだが大きく開いたオープン型の装置が多くあります。閉所が苦手な患者さんや、検査しながらの処置に向きます。最近は0.55 Tの超伝導装置も使われ始めています
- **1.5 T**：現在も広く使われている標準的な装置です
- **3 T**：主に脳、関節、前立腺などの細かい構造を見る検査で使われます
- **7 T以上（超高磁場）**：7 Tの装置は2017年に頭部と四肢の臨床用として承認されました。研究では11.7 Tの人体用装置も使われ始めています

1.5 Tと3 Tを比べると、磁場の強さを2倍にすることで、次のような違いが生じます。

| 項目 | 1.5 T | 3 T | 理由 |
|---|---|---|---|
| ラーモア周波数 | 約64 MHz | 約128 MHz | 周波数は磁場に比例する（1-1） |
| SN比 | 1 | 約2倍 | 磁場の方向にそろう原子核の差も、コイルに生じる電圧も磁場とともに増える |
| T1 | 短め | 長くなる | 組織どうしのT1の差が縮み、T1強調画像のコントラストはやや下がる。TOF（1-5）では背景がより飽和しやすくなり、血管が見やすくなる |
| 化学シフト（1-2） | 約220 Hz | 約440 Hz | ppmで決まる周波数の差が、磁場に比例して大きくなる |
| 金属や空気による磁場の乱れ | 小さい | 約2倍 | 物質ごとの磁化されやすさ（磁化率）の違いで生じる磁場のずれは、磁場に比例する。金属や空気の近くの歪みやEPIの歪みは増えるが、T2*強調画像やfMRIの感度は上がる |
| SAR（1-1） | 1 | 約4倍 | RFパルスの周波数が高くなり、体が吸収するエネルギーは磁場のおよそ2乗に比例する |
| 体内の電波の波長 | 約50 cm | 約30 cm | 波長が体の大きさに近くなり、腹部などで電波の強さにむらが生じて、画像の一部が暗くなる |

3 Tは、増えたSN比を、画素を小さくすることや撮影時間の短縮（1-7のパラレルイメージングとの組み合わせなど）に回せるのが大きな利点です。一方で、SARの制限のために高速スピンエコー法の条件が制約されたり、体内の金属や医療機器の安全性が1.5 Tとは別に評価されていたりするので、検査の目的と患者さんに合わせて装置を選びます。低磁場の装置は、SN比が低い分をディープラーニングによる再構成（1-3のノートブック）などで補い、SARや金属の影響が小さいこと、費用が低いことを生かす方向で見直されています。

![Siemens MAGNETOM Skyra 3 T MRI system](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/photo_mri_3t.jpg)

3 TのMRI装置の例（Siemens Healthineers MAGNETOM Skyra、台湾の国立政治大学の脳画像研究センター）です。外観は1-1の1.5 Tの装置と大きくは変わりませんが、中の超伝導磁石はより強い磁場を作ります。寝台の奥のボアの入口に、頭部用のコイルが置かれています。

出典: Wikimedia Commons — [Siemens MAGNETOM Skyra 3 Tesla at Taiwan Mind & Brain Imaging Center 20241120.jpg](https://commons.wikimedia.org/wiki/File:Siemens_MAGNETOM_Skyra_3_Tesla_at_Taiwan_Mind_%26_Brain_Imaging_Center_20241120.jpg) ／ 撮影: Yu tptw ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0)

### 1-9. 超音波と核医学の原理（概要）

**超音波**は、体に当てた音波が組織の境界で跳ね返ってくるまでの時間から深さを、跳ね返りの強さから明るさを決めて画像にします。装置が小さくその場で動きを観察できる一方、骨やガスの奥は音波が届かず見えにくくなります。

**核医学**（PET、SPECT）は、放射性物質で目印を付けた薬剤を投与し、その薬剤が体のどこに集まったかを画像にします。たとえばPETでよく使うFDGはブドウ糖に似た薬剤で、糖を多く取り込む組織（多くのがんなど）に集まります。臓器の形より「働き」を見る検査で、解像度が低いため、CTと組み合わせたPET-CTとして使われることが多くなっています。

## 2. 動かしてみる

教員が用意したコードを、上から順に実行します。

### 2-1. T1強調画像とT2強調画像を並べる

2つのシリーズを読み込み、腰椎と腸管が写る同じくらいの高さのスライスを並べます。背骨の中の脳脊髄液（水）と、皮下の脂肪の明るさに注目してください。

In [ ]:
def load_series(folder):
    """フォルダ内のDICOMファイルを読み込み、スライスの位置（z座標）の順に並べる"""
    slices = [pydicom.dcmread(f) for f in folder.glob("*.dcm")]
    slices.sort(key=lambda ds: float(ds.ImagePositionPatient[2]))
    return slices

t2_slices = load_series(fetch("mr"))
t1_slices = load_series(fetch("mr_t1"))
print("T2:", len(t2_slices), "枚", " T1:", len(t1_slices), "枚")

# 2つのシリーズは撮影の範囲が違うので、画像を見て同じくらいの高さのスライスを選んである
t2 = t2_slices[25].pixel_array.astype(float)
t1 = t1_slices[4].pixel_array.astype(float)

for name, ds in [("T2 (HASTE)", t2_slices[25]), ("T1 (FLASH)", t1_slices[4])]:
    print(f"{name}: TR={float(ds.RepetitionTime):.0f} ms, TE={float(ds.EchoTime):.1f} ms")
print("T2 の画素値の範囲:", t2.min(), "〜", t2.max(), " T1 の画素値の範囲:", t1.min(), "〜", t1.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(t2, cmap="gray")
axes[0].set_title("T2-weighted (HASTE)")
axes[1].imshow(t1, cmap="gray")
axes[1].set_title("T1-weighted (FLASH)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

撮影に使った装置の条件も、DICOMのタグに記録されています。磁場の強さと共鳴周波数から1-1の式 $f_0 = \bar\gamma B_0$ を、位相エンコードの方向から1-2の説明を確かめます。

In [ ]:
ds = t2_slices[25]
b0 = float(ds.MagneticFieldStrength)  # 静磁場の強さ [T]
f0 = float(ds.ImagingFrequency)       # 共鳴周波数 [MHz]
print(f"磁場の強さ: {b0:.3f} T")
print(f"共鳴周波数: {f0:.3f} MHz")
print(f"共鳴周波数 ÷ 磁場の強さ: {f0 / b0:.2f} MHz/T")  # 42.58 MHz/T に近くなるはず
print("位相エンコードの方向:", ds.InPlanePhaseEncodingDirection)  # ROW は左右、COL は上下
print("受信コイル:", ds.get("ReceiveCoilName", "記録なし"))

### 2-2. 画像をフーリエ変換してk空間を表示する

`np.fft.fft2` で画像をフーリエ変換し、`np.fft.fftshift` で低い周波数の成分が中心に来るように並べ替えます。成分の大きさは桁が大きく違うので、対数をとって表示します。

In [ ]:
image = t2
kspace = np.fft.fftshift(np.fft.fft2(image))


def to_image(k):
    """k空間を画像に戻す（並べ替えを元に戻してから逆フーリエ変換し、絶対値をとる）"""
    return np.abs(np.fft.ifft2(np.fft.ifftshift(k)))


fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Image")
axes[1].imshow(np.log1p(np.abs(kspace)), cmap="gray")
axes[1].set_title("k-space (log magnitude)")
axes[2].imshow(to_image(kspace), cmap="gray")
axes[2].set_title("Back to image")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("元の画像と、戻した画像の最大の差:", np.max(np.abs(to_image(kspace) - image)))

### 2-3. k空間の中心だけを残して画像に戻す

k空間の中心から半径16画素の円の内側だけを残し、外側を0にしてから画像に戻します。

In [ ]:
ny, nx = image.shape
y, x = np.ogrid[:ny, :nx]
distance = np.hypot(y - ny // 2, x - nx // 2)  # k空間の中心からの距離
RADIUS = 16

center_only = kspace * (distance <= RADIUS)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(np.log1p(np.abs(center_only)), cmap="gray")
axes[1].set_title(f"k-space: center only (r <= {RADIUS})")
axes[2].imshow(to_image(center_only), cmap="gray")
axes[2].set_title("Image from center only")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 2-4. k空間の周辺だけを残して画像に戻す

今度は逆に、中心の円の内側を0にして、周辺だけを残します。

In [ ]:
periphery_only = kspace * (distance > RADIUS)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(np.log1p(np.abs(periphery_only)), cmap="gray")
axes[1].set_title(f"k-space: periphery only (r > {RADIUS})")
axes[2].imshow(to_image(periphery_only), cmap="gray")
axes[2].set_title("Image from periphery only")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. AIに頼んでみる

頼む前に、ソース管理（`Ctrl+Shift+G`）でここまでの状態をコミットします。AIが変更した後は、差分を見て採用するかどうかを決めます。頼み方は `setup/03_ai_assistant.md` を参照してください。

発展課題は、基本課題を終えた人が取り組みます。

### 基本課題1：残す範囲を変える

k空間の中心から残す半径を 4、16、64 の3通りに変えて画像に戻し、並べて表示するよう頼みます。

### 基本課題2：結果の意味を聞く

2-3と2-4の画像の違いが何を意味するかを、解説役（`tutor`）に質問します。答えを読んだら、自分の言葉で「k空間の中心は画像の〇〇を、周辺は〇〇を担っている」とまとめ、振り返りに書きます。

### 発展課題1：k空間を間引く

k空間を1行おきに間引いて（間引いた行を0にして）から画像に戻し、画像に何が起きるか確かめます。MRIでは、収集する行を減らすと撮影時間を短くできます。その代わりに何が起きるかを考えます。

## 4. AIの答えを確かめる

確認できた項目は `[ ]` を `[x]` に書き換えます。

- [ ] 2-1で、脳脊髄液（水）がT2強調画像で明るく、T1強調画像で暗いことを確認した
- [ ] k空間を表示するとき、低い周波数が中心に来るように並べ替えている（`np.fft.fftshift`）
- [ ] 画像に戻したとき、複素数の絶対値を表示している
- [ ] 残す半径と画像の変化の関係を、図と文章で記録した

## 5. 振り返り

| 項目 | 記入欄 |
|---|---|
| 使ったプロンプト | |
| AIの答えで直した点・採用しなかった点 | |
| この回で分かったこと | |
| まだ分からないこと | |

記入したら保存してコミットします。提出のしかたは授業で指示します。